# 📰 NewsDesk Pro — Self-Graded Lab (Medium → Hard)

You run the NLP desk for a newsroom. You'll tokenise text properly, pull out named entities, redact private names, clean and lemmatise messy text, parse a server log, and then master three prompting styles — **zero-shot**, **few-shot**, and **chain-of-thought** — on harder tasks.

| Rank | Score |
|---|---|
| 📰 Cub Reporter | 0–25% |
| ✂️ Copy Editor | 25–45% |
| 🔍 NLP Analyst | 45–65% |
| 🧠 Prompt Engineer | 65–85% |
| 🦅 NewsDesk Chief | 85–100% |

**12 tasks · 120 points.** Some tasks use real NLP models (spaCy, BERT) — they download once, then run for real. Grading is forgiving on model wording but strict on your logic. Hints hide in **💡 dropdowns**.


## ⚙️ Setup 1 — Install the NLP stack
This downloads small models the first time (a minute or two). Everything here is standard in Colab.

In [ ]:
!pip install -q langchain-groq transformers spacy nltk
!python -m spacy download en_core_web_md

## ⚙️ Setup 2 — Optional live model
Only needed for the "Try it live" prompting cells. Add your Groq key in Colab **Secrets** as `groq_api`.

In [ ]:
try:
    from google.colab import userdata
    from langchain_groq import ChatGroq
    llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, api_key=userdata.get('groq_api'))
    print("LLM ready for live demos.")
except Exception as e:
    llm = None
    print("LLM not configured (grading still works). Detail:", e)


## ⚙️ Setup 3 — Grader + a sample log file *(just run it)*

In [ ]:
def has_any(text, words):
    t=(text or "").lower(); return any(w in t for w in words)
def has_all(text, words):
    t=(text or "").lower(); return all(w in t for w in words)

# create a server.log file so Task 8 has something real to read
log_contents = """2026-02-01 09:00 INFO Service_Started
2026-02-01 09:03 ERROR Database_Timeout
2026-02-01 09:05 INFO Cache_Refreshed
2026-02-01 09:09 ERROR Auth_Failed
2026-02-01 09:12 WARN High_Latency
2026-02-01 09:15 ERROR Disk_Full"""
with open("server.log", "w") as f:
    f.write(log_contents)

class Grader:
    RANKS=[(0.00,"📰 Cub Reporter","Fresh on the desk."),
           (0.25,"✂️ Copy Editor","You clean text like a pro."),
           (0.45,"🔍 NLP Analyst","Entities cannot hide from you."),
           (0.65,"🧠 Prompt Engineer","You bend LLMs to your will."),
           (0.85,"🦅 NewsDesk Chief","The whole desk answers to you.")]
    def __init__(self,total=120): self.s={}; self.m={}; self.total=total
    @property
    def earned(self): return sum(self.s.values())
    def _bar(self,p,w=30): f=int(round(p*w)); return "#"*f+"-"*(w-f)
    def _rank(self):
        p=self.earned/self.total; r=self.RANKS[0]
        for t,n,b in self.RANKS:
            if p>=t: r=(t,n,b)
        return r
    def check(self,tid,passed,pts,ok="",fail=""):
        self.m[tid]=pts; done=self.s.get(tid,0)==pts
        if passed:
            self.s[tid]=pts
            print(("Already done" if done else "PASS  +"+str(pts)+" pts")+"  [Task "+tid+"]")
            if ok and not done: print("   "+ok)
        else:
            if not done: self.s[tid]=0
            print("Not yet  (0/"+str(pts)+")  [Task "+tid+"]")
            if fail: print("   "+fail)
        p=self.earned/self.total; _,n,b=self._rank()
        print(); print("   "+str(self.earned)+"/"+str(self.total)+"  ["+self._bar(p)+"]  "+str(round(p*100))+"%")
        print("   "+n+" - "+b); print()
    def scorecard(self):
        p=self.earned/self.total; _,n,b=self._rank()
        print("="*50); print("  FINAL SCORECARD"); print("="*50)
        for tid in sorted(self.m,key=lambda x:(len(x),x)):
            e,t=self.s.get(tid,0),self.m[tid]
            print("  "+("[x]" if e==t else ("[~]" if e>0 else "[ ]"))+"  Task "+tid.ljust(4)+" "+str(e).rjust(3)+"/"+str(t))
        print("  "+"-"*40); print("  TOTAL  "+str(self.earned).rjust(3)+"/"+str(self.total)+"  ["+self._bar(p)+"]  "+str(round(p*100))+"%")
        print("  "+n+"\n  "+b); print("="*50)
grader=Grader(120)
print("Grader ready. 120 points, 12 tasks. (server.log created)")


---
## TASK 1 — Write a reusable function &nbsp;`8 pts`

Functions let you reuse logic. Write **`token_count(text)`** that returns the **number of words** in `text` (split on spaces). It must work for any string.


In [ ]:
# TASK 1 — def token_count(text): ...


<details><summary>💡 Hint</summary>

```python
def token_count(text):
    return len(text.split())
```
</details>

In [ ]:
# ✅ CHECK 1
try:
    ok = token_count("hello world")==2 and token_count("one two three four")==4 and token_count("single")==1
    grader.check("1", ok, 8, ok="token_count works for any text.",
        fail="token_count('one two three four') should be 4. Got "+str(token_count('one two three four') if 'token_count' in dir() else '?'))
except NameError:
    grader.check("1", False, 8, fail="`token_count` not defined.")
except Exception as e:
    grader.check("1", False, 8, fail="Error: "+str(e))


---
## TASK 2 — Subword tokenisation with BERT &nbsp;`8 pts`

Splitting on spaces is crude. Real LLMs use **subword** tokens (1 token ≈ ¾ of a word), so long/rare words get broken into pieces. Load the BERT tokenizer and tokenise `headline`. Store the result in **`bert_tokens`**.


In [ ]:
# TASK 2
headline = "The decentralization of hyperscale infrastructure was unprecedented"

# Load the pretrained tokenizer and tokenise the headline into bert_tokens


<details><summary>💡 Hint</summary>

```python
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_tokens = tokenizer.tokenize(headline)
print(bert_tokens)
```
Notice long words split into pieces marked with `##`.
</details>

In [ ]:
# ✅ CHECK 2
try:
    is_list = isinstance(bert_tokens, list) and len(bert_tokens) > 0 and all(isinstance(t, str) for t in bert_tokens)
    # subwords mean MORE tokens than a plain word-split on this long-word sentence
    more_than_split = len(bert_tokens) > len(headline.split())
    grader.check("2", is_list and more_than_split, 8,
        ok="Got "+str(len(bert_tokens))+" subword tokens vs "+str(len(headline.split()))+" words - subwords at work.",
        fail="bert_tokens should be a non-empty list with more tokens than words. Got "+repr(globals().get('bert_tokens'))[:120])
except NameError:
    grader.check("2", False, 8, fail="`bert_tokens` not defined.")
except Exception as e:
    grader.check("2", False, 8, fail="Error: "+str(e))


---
## TASK 3 — Find the named entities &nbsp;`10 pts`

Load spaCy's English model, run it on `article`, and collect every entity's text into a list **`entity_texts`**.


In [ ]:
# TASK 3
article = """Microsoft chief Satya Nadella met Sundar Pichai in London on Tuesday.
They discussed a new data centre in Mumbai worth two billion dollars."""

# Load spacy 'en_core_web_md', run it on the article, collect entity texts into entity_texts


<details><summary>💡 Hint</summary>

```python
import spacy
nlp = spacy.load('en_core_web_md')
doc = nlp(article)
entity_texts = [ent.text for ent in doc.ents]
print(entity_texts)
```
</details>

In [ ]:
# ✅ CHECK 3
try:
    ok = isinstance(entity_texts, list) and len(entity_texts) >= 3
    grader.check("3", ok, 10, ok="Found entities: "+str(entity_texts),
        fail="entity_texts should be a list of the entities found (expect a few). Got "+repr(globals().get('entity_texts'))[:120])
except NameError:
    grader.check("3", False, 10, fail="`entity_texts` not defined.")
except Exception as e:
    grader.check("3", False, 10, fail="Error: "+str(e))


---
## TASK 4 — Redact the names (privacy bot) &nbsp;`12 pts`

Privacy first: replace every **PERSON** entity in `article` with `[REDACTED]`. Build the result into **`clean_text`**. (Loop the entities, and when the label is `PERSON`, use `.replace(...)`.)


In [ ]:
# TASK 4 — start from the article and redact PERSON names into clean_text


<details><summary>💡 Hint</summary>

```python
clean_text = article
doc = nlp(article)
for ent in doc.ents:
    if ent.label_ == "PERSON":
        clean_text = clean_text.replace(ent.text, "[REDACTED]")
print(clean_text)
```
</details>

In [ ]:
# ✅ CHECK 4
try:
    redacted = "[REDACTED]" in clean_text
    names_gone = ("Satya Nadella" not in clean_text) and ("Sundar Pichai" not in clean_text)
    grader.check("4", redacted and names_gone, 12,
        ok="Names hidden behind [REDACTED].",
        fail="Expected [REDACTED] present and the person names removed. redacted="+str(redacted)+", names_gone="+str(names_gone))
except NameError:
    grader.check("4", False, 12, fail="`clean_text` not defined.")
except Exception as e:
    grader.check("4", False, 12, fail="Error: "+str(e))


---
## TASK 5 — Remove the stopwords &nbsp;`8 pts`

Stopwords (the, is, a, ...) carry little meaning. Run spaCy on `sentence` and build **`content_words`** — a list of the **token texts that are NOT stopwords**.


In [ ]:
# TASK 5
sentence = "The editor is reviewing the article about the new policy"

# Build content_words: the non-stopword token texts


<details><summary>💡 Hint</summary>

```python
doc = nlp(sentence)
content_words = [token.text for token in doc if not token.is_stop]
print(content_words)
```
</details>

In [ ]:
# ✅ CHECK 5
try:
    low = [w.lower() for w in content_words]
    removed = ("the" not in low) and ("is" not in low)
    kept = any(w in low for w in ["editor","reviewing","article","policy"])
    grader.check("5", isinstance(content_words,list) and removed and kept, 8,
        ok="Stopwords removed: "+str(content_words),
        fail="'the'/'is' should be gone and content words kept. Got "+repr(globals().get('content_words'))[:120])
except NameError:
    grader.check("5", False, 8, fail="`content_words` not defined.")
except Exception as e:
    grader.check("5", False, 8, fail="Error: "+str(e))


---
## TASK 6 — Lemmatise to root words &nbsp;`10 pts`

Different word forms (running, ran, runs) should map to one root. Run spaCy on `messy` and build **`lemmas`** — a list of each token's **lemma** (`token.lemma_`).


In [ ]:
# TASK 6
messy = "The batteries were cracked and the cables are fraying"

# Build lemmas: the lemma_ of each token


<details><summary>💡 Hint</summary>

```python
doc = nlp(messy)
lemmas = [token.lemma_ for token in doc]
print(lemmas)
```
</details>

In [ ]:
# ✅ CHECK 6
try:
    low = [l.lower() for l in lemmas]
    ok = isinstance(lemmas,list) and ("battery" in low) and ("cable" in low)
    grader.check("6", ok, 10, ok="Lemmatised correctly (batteries->battery, cables->cable).",
        fail="Expected 'battery' and 'crack' among the lemmas. Got "+repr(globals().get('lemmas'))[:140])
except NameError:
    grader.check("6", False, 10, fail="`lemmas` not defined.")
except Exception as e:
    grader.check("6", False, 10, fail="Error: "+str(e))


---
## TASK 7 — The keyword extractor &nbsp;`12 pts`

Combine the last two ideas into one reusable function. Write **`extract_keywords(text)`** that returns a list of **lemmas of the tokens that are NOT stopwords and NOT punctuation**.


In [ ]:
# TASK 7 — def extract_keywords(text): ...


<details><summary>💡 Hint</summary>

```python
def extract_keywords(text):
    doc = nlp(text)
    keywords = []
    for token in doc:
        if not token.is_stop and not token.is_punct:
            keywords.append(token.lemma_)
    return keywords
```
</details>

In [ ]:
# ✅ CHECK 7
try:
    out = extract_keywords("My batteries aren't charging properly and the screen is cracked!")
    low = [w.lower() for w in out]
    ok = ("battery" in low) and ("crack" in low or "cracked" in low) and ("the" not in low) and ("!" not in out)
    grader.check("7", ok, 12, ok="Keywords extracted (stopwords + punctuation removed, lemmatised): "+str(out),
        fail="Expected 'battery' and 'crack', no 'the' or punctuation. Got "+repr(out)[:140])
except NameError:
    grader.check("7", False, 12, fail="`extract_keywords` not defined.")
except Exception as e:
    grader.check("7", False, 12, fail="Error: "+str(e))


---
## TASK 8 — Parse the server log &nbsp;`12 pts`

A file `server.log` already exists (created in setup). **Read it**, go line by line, and for every line that contains `"ERROR"`, append the **timestamp** (`date time`, i.e. the first two words) to **`error_times`**. Use `strip()` and `split()`.


In [ ]:
# TASK 8 — read server.log and build error_times


<details><summary>💡 Hint</summary>

```python
error_times = []
with open("server.log", "r") as f:
    for line in f.readlines():
        clean = line.strip()
        if "ERROR" in clean:
            parts = clean.split()
            error_times.append(f"{parts[0]} {parts[1]}")
print(error_times)
```
</details>

In [ ]:
# ✅ CHECK 8
try:
    exp = ["2026-02-01 09:03", "2026-02-01 09:09", "2026-02-01 09:15"]
    grader.check("8", error_times == exp, 12, ok="Found "+str(len(error_times))+" errors with timestamps.",
        fail="Expected "+str(exp)+"\n   Got "+repr(globals().get('error_times')))
except NameError:
    grader.check("8", False, 12, fail="`error_times` not defined.")
except Exception as e:
    grader.check("8", False, 12, fail="Error: "+str(e))


---
# 💬 Prompting, harder edition

Same three styles, tougher jobs. You build the prompt strings; grading checks they're built right; the optional cells run them for real.


---
## TASK 9 — Zero-shot with a strict format &nbsp;`8 pts`

Build **`zero_shot_prompt`** that asks the model to read `headline_text` and return **only** the main organisation mentioned — no explanation. Include the headline (f-string) and demand a one-word / name-only answer.


In [ ]:
# TASK 9
headline_text = "Tesla recalls 200,000 vehicles over a software glitch, shares dip."

# Build zero_shot_prompt


<details><summary>💡 Hint</summary>

```python
zero_shot_prompt = f"Read the headline and reply with ONLY the main company name, nothing else.\nHeadline: {headline_text}\nCompany:"
```
</details>

In [ ]:
# ✅ CHECK 9
try:
    ok = headline_text in zero_shot_prompt and has_any(zero_shot_prompt, ["only","company","organisation","organization","name"]) and len(zero_shot_prompt) > 40
    grader.check("9", ok, 8, ok="Zero-shot prompt is direct and includes the headline.",
        fail="Include the headline AND ask for only the company/name. Got: "+repr(globals().get('zero_shot_prompt'))[:120])
except NameError:
    grader.check("9", False, 8, fail="`zero_shot_prompt` not defined.")


### ▶️ Try it live *(optional)*

In [ ]:
try:
    print(llm.invoke(zero_shot_prompt).content)
except Exception as e:
    print("Add your API key to run this. Detail:", e)


---
## TASK 10 — Few-shot category tagging &nbsp;`12 pts`

You want to tag headlines as `Tech`, `Sports`, or `Finance`. Teach by example: build **`few_shot_prompt`** by **looping** over `examples` (list of dicts), then add `new_headline` with a trailing `Category:`.


In [ ]:
# TASK 10
examples = [
    {"headline": "Apple unveils a faster M-series chip",        "category": "Tech"},
    {"headline": "City FC wins the league on the final day",    "category": "Sports"},
    {"headline": "Central bank holds interest rates steady",    "category": "Finance"},
]
new_headline = "Start-up raises 50 million in a new funding round"


<details><summary>💡 Hint</summary>

```python
few_shot_prompt = "Tag each headline as Tech, Sports, or Finance.\n\n"
for ex in examples:
    few_shot_prompt += f"Headline: {ex['headline']}\nCategory: {ex['category']}\n\n"
few_shot_prompt += f"Headline: {new_headline}\nCategory:"
```
</details>

In [ ]:
# ✅ CHECK 10
try:
    p = few_shot_prompt
    labels = has_all(p, ["tech","sports","finance"])
    ex_in = all(ex["headline"] in p for ex in examples)
    q_in = new_headline in p
    grader.check("10", labels and ex_in and q_in, 12,
        ok="Few-shot prompt has all 3 labelled examples + the new headline.",
        fail="labels_ok="+str(labels)+", examples_in="+str(ex_in)+", query_in="+str(q_in))
except NameError:
    grader.check("10", False, 12, fail="`few_shot_prompt` not defined.")


### ▶️ Try it live *(optional)*

In [ ]:
try:
    print(llm.invoke(few_shot_prompt).content)
except Exception as e:
    print("Add your API key to run this. Detail:", e)


---
## TASK 11 — Chain-of-thought on a multi-step problem &nbsp;`12 pts`

Reasoning problems need working-out. Build **`cot_prompt`** that gives the model `problem` and explicitly tells it to **reason step by step** before stating the final answer.


In [ ]:
# TASK 11
problem = ("A newsroom published 12 articles on Monday, twice as many on Tuesday, "
           "and 5 fewer on Wednesday than Tuesday. How many articles in total over the three days?")


<details><summary>💡 Hint</summary>

```python
cot_prompt = f"{problem}\nWork through it step by step, then give the final total."
```
</details>

In [ ]:
# ✅ CHECK 11
try:
    ok = (problem in cot_prompt
          and (has_any(cot_prompt, ["step by step","step-by-step"]) or has_all(cot_prompt,["step","reason"]) or has_all(cot_prompt,["work","step"]))
          and len(cot_prompt) > len(problem))
    grader.check("11", ok, 12, ok="Chain-of-thought prompt asks for step-by-step reasoning.",
        fail="Include the problem AND ask it to reason step by step. Got: "+repr(globals().get('cot_prompt'))[:140])
except NameError:
    grader.check("11", False, 12, fail="`cot_prompt` not defined.")


### ▶️ Try it live *(optional)* — see the reasoning unfold

In [ ]:
try:
    print(llm.invoke(cot_prompt).content)
except Exception as e:
    print("Add your API key to run this. Detail:", e)


---
## TASK 12 — When do you use which? &nbsp;`8 pts`

Match each situation to the best prompting style. Fill **`technique_for`** using exactly these labels: `"zero_shot"`, `"few_shot"`, `"chain_of_thought"`.

- `"quick_question_no_examples"` — a simple ask where the model already knows the answer
- `"teach_a_specific_label_format"` — you need outputs in your exact categories/format
- `"multi_step_math_or_logic"` — a problem that needs working-out to get right


In [ ]:
# TASK 12 — technique_for = { ... }


<details><summary>💡 Hint</summary>

Think about what each style is *for*: just ask, show examples, or make it reason.
</details>

In [ ]:
# ✅ CHECK 12
key = {"quick_question_no_examples":"zero_shot",
       "teach_a_specific_label_format":"few_shot",
       "multi_step_math_or_logic":"chain_of_thought"}
try:
    got = {k: str(v).strip().lower() for k,v in technique_for.items()}
    wrong = [k for k in key if got.get(k)!=key[k]]
    grader.check("12", len(wrong)==0, 8, ok="You know which prompting style fits which job.",
        fail="Re-check these: "+str(wrong))
except NameError:
    grader.check("12", False, 8, fail="`technique_for` not defined.")
except Exception as e:
    grader.check("12", False, 8, fail="Error: "+str(e))


---
## 🏆 Final Scorecard

In [ ]:
grader.scorecard()

## 🎓 Recap
You ran a real NLP pipeline — subword tokenisation, named-entity recognition, redaction, stopword removal, lemmatisation, a reusable keyword extractor, and a log parser — then applied **zero-shot**, **few-shot**, and **chain-of-thought** prompting to harder problems and learned when each one wins.
